# 01 — Dataset overview

Validate the authoritative physiology–MWL modeling dataset before statistical analysis. This notebook reports potential issues but does not remove, transform, or impute features.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Paths and configuration

The root resolver supports execution from either the repository root or `postprocessing/`.

In [ ]:
SAVE_FIGURES = True
SAVE_TABLES = True
NEAR_ZERO_VARIANCE_THRESHOLD = 1e-12
GROSS_MAGNITUDE_THRESHOLD = 1e12

def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        expected = candidate / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
        if expected.exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory")

REPO_ROOT = find_repository_root()
INPUT_PATH = REPO_ROOT / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
FIGURE_DIR = REPO_ROOT / "postprocessing/outputs/figures"
TABLE_DIR = REPO_ROOT / "postprocessing/outputs/tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Input:", INPUT_PATH)

## 3. Load and validate the authoritative dataset

In [ ]:
data = pd.read_csv(INPUT_PATH)
KEY_COLUMNS = ["participant_id", "phase", "block_index"]
FEATURE_PREFIXES = ("delta_ecg_", "delta_eda_", "delta_resp_", "delta_temp_", "fnirs_")
feature_columns = [column for column in data.columns if column.startswith(FEATURE_PREFIXES)]
modality_features = {
    "ECG": [c for c in feature_columns if c.startswith("delta_ecg_")],
    "EDA": [c for c in feature_columns if c.startswith("delta_eda_")],
    "RESP": [c for c in feature_columns if c.startswith("delta_resp_")],
    "TEMP": [c for c in feature_columns if c.startswith("delta_temp_")],
    "fNIRS": [c for c in feature_columns if c.startswith("fnirs_")],
}

assert not data.duplicated(KEY_COLUMNS).any(), "Duplicate participant/phase/block keys"
assert data["mwl_value"].notna().all(), "The final analysis dataset contains missing MWL"
assert len(feature_columns) == 79, f"Expected 79 physiological features, found {len(feature_columns)}"
assert sum(map(len, modality_features.values())) == 79
print(f"Loaded {len(data)} rows with {len(data.columns)} columns and {len(feature_columns)} physiological features.")

## 4. Dimensions

In [ ]:
dimension_summary = pd.DataFrame({
    "metric": ["rows", "columns", "participants", "groups", "phases", "physiological_features"],
    "value": [len(data), len(data.columns), data.participant_id.nunique(), data.group.nunique(), data.phase.nunique(), len(feature_columns)],
})
feature_count_summary = pd.Series({m: len(c) for m, c in modality_features.items()}, name="n_features").rename_axis("modality").reset_index()
print(dimension_summary.to_string(index=False))
print("\nFeature counts:\n", feature_count_summary.to_string(index=False))
if SAVE_TABLES:
    dimension_summary.to_csv(TABLE_DIR / "01_dimension_summary.csv", index=False)
    feature_count_summary.to_csv(TABLE_DIR / "01_feature_count_summary.csv", index=False)

## 5. Observation counts

In [ ]:
counts_by_participant = data.groupby(["participant_id", "group"], observed=True).size().rename("n_observations").reset_index()
counts_by_group = data.groupby("group", observed=True).size().rename("n_observations").reset_index()
counts_by_phase = data.groupby("phase", observed=True).size().rename("n_observations").reset_index()
counts_participant_phase = data.pivot_table(index=["participant_id", "group"], columns="phase", values="block_index", aggfunc="count", fill_value=0)

print("By group:\n", counts_by_group.to_string(index=False))
print("\nBy phase:\n", counts_by_phase.to_string(index=False))
print("\nBy participant × phase:\n", counts_participant_phase.to_string())
if SAVE_TABLES:
    counts_by_participant.to_csv(TABLE_DIR / "01_counts_by_participant.csv", index=False)
    counts_by_group.to_csv(TABLE_DIR / "01_counts_by_group.csv", index=False)
    counts_by_phase.to_csv(TABLE_DIR / "01_counts_by_phase.csv", index=False)
    counts_participant_phase.to_csv(TABLE_DIR / "01_counts_participant_phase.csv")

## 6. MWL availability and provenance

In [ ]:
mwl_source_counts = data.groupby("mwl_source", dropna=False).size().rename("n_observations").reset_index()
imputed_mwl = data.loc[data.mwl_source.ne("observed"), ["participant_id", "group", "phase", "block_index", "mwl_value", "mwl_source"]]
print("Missing MWL values:", int(data.mwl_value.isna().sum()))
print(mwl_source_counts.to_string(index=False))
print("\nApproved non-observed observation:\n", imputed_mwl.to_string(index=False))
if SAVE_TABLES:
    mwl_source_counts.to_csv(TABLE_DIR / "01_mwl_source_counts.csv", index=False)
    imputed_mwl.to_csv(TABLE_DIR / "01_imputed_mwl_observation.csv", index=False)

## 7. Physiological missingness

Missingness is reported, not imputed. Cell-level fractions and blocks with an entirely absent modality are kept distinct.

In [ ]:
feature_missingness = pd.DataFrame({
    "feature_name": feature_columns,
    "missing_count": data[feature_columns].isna().sum().to_numpy(),
    "missing_fraction": data[feature_columns].isna().mean().to_numpy(),
})
modality_missingness = []
for modality, columns in modality_features.items():
    modality_missingness.append({
        "modality": modality,
        "n_features": len(columns),
        "missing_cells": int(data[columns].isna().sum().sum()),
        "missing_cell_fraction": float(data[columns].isna().mean().mean()),
        "blocks_with_any_missing_feature": int(data[columns].isna().any(axis=1).sum()),
        "blocks_with_modality_entirely_missing": int(data[columns].isna().all(axis=1).sum()),
    })
modality_missingness = pd.DataFrame(modality_missingness)
participant_missingness = data.groupby(["participant_id", "group"], observed=True)[feature_columns].apply(lambda x: x.isna().mean().mean()).rename("missing_cell_fraction").reset_index()
phase_missingness = data.groupby("phase", observed=True)[feature_columns].apply(lambda x: x.isna().mean().mean()).rename("missing_cell_fraction").reset_index()
print(modality_missingness.to_string(index=False))
print("\nMissingness by phase:\n", phase_missingness.to_string(index=False))
if SAVE_TABLES:
    feature_missingness.to_csv(TABLE_DIR / "01_feature_missingness.csv", index=False)
    modality_missingness.to_csv(TABLE_DIR / "01_modality_missingness.csv", index=False)
    participant_missingness.to_csv(TABLE_DIR / "01_participant_missingness.csv", index=False)
    phase_missingness.to_csv(TABLE_DIR / "01_phase_missingness.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(modality_missingness.modality, 100 * modality_missingness.missing_cell_fraction)
ax.set(ylabel="Missing feature cells [%]", title="Physiological missingness by modality")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "01_missingness_by_modality.png", dpi=160)
plt.show()

## 8. Block and modality coverage

In [ ]:
trial_count_distribution = data.groupby("n_trials_available").size().rename("n_blocks").reset_index()
block_status_counts = data.groupby("block_status", dropna=False).size().rename("n_blocks").reset_index()
availability_columns = ["n_ecg_trials", "n_eda_trials", "n_resp_trials", "n_temp_trials", "n_fnirs_trials"]
modality_trial_coverage = data[["participant_id", "phase", "block_index", "n_trials_available", *availability_columns]].copy()
modality_trial_coverage["has_partial_modality_coverage"] = modality_trial_coverage[availability_columns].lt(modality_trial_coverage.n_trials_available, axis=0).any(axis=1)
print("Trials contributing to blocks:\n", trial_count_distribution.to_string(index=False))
print("\nBlock status:\n", block_status_counts.to_string(index=False))
print("\nBlocks with partial modality coverage:", int(modality_trial_coverage.has_partial_modality_coverage.sum()))
if SAVE_TABLES:
    trial_count_distribution.to_csv(TABLE_DIR / "01_trial_count_distribution.csv", index=False)
    block_status_counts.to_csv(TABLE_DIR / "01_block_status_counts.csv", index=False)
    modality_trial_coverage.to_csv(TABLE_DIR / "01_modality_trial_coverage.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(trial_count_distribution.n_trials_available.astype(str), trial_count_distribution.n_blocks)
ax.set(xlabel="Available physiological trials per block", ylabel="Number of blocks", title="Block coverage")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "01_block_coverage.png", dpi=160)
plt.show()

## 9. Feature sanity checks

The gross-magnitude check is deliberately generic. It flags numerical scale issues without imposing unverified physiological ranges on derived features.

In [ ]:
feature_values = data[feature_columns].astype(float)
infinite_counts = pd.Series(np.isinf(feature_values.to_numpy()).sum(axis=0), index=feature_columns)
feature_variances = feature_values.var(skipna=True)
feature_unique_counts = feature_values.nunique(dropna=True)
constant_features = feature_unique_counts[feature_unique_counts <= 1].index.tolist()
near_zero_variance_features = feature_variances[(feature_variances > 0) & (feature_variances <= NEAR_ZERO_VARIANCE_THRESHOLD)].index.tolist()
gross_magnitude = feature_values.abs().max(skipna=True)
gross_magnitude_features = gross_magnitude[gross_magnitude > GROSS_MAGNITUDE_THRESHOLD].index.tolist()

sanity = pd.DataFrame({
    "feature_name": feature_columns,
    "non_missing_n": feature_values.notna().sum().to_numpy(),
    "infinite_n": infinite_counts.to_numpy(),
    "variance": feature_variances.to_numpy(),
    "unique_non_missing": feature_unique_counts.to_numpy(),
    "maximum_absolute_value": gross_magnitude.to_numpy(),
})
sanitary_issues = sanity[(sanity.infinite_n > 0) | (sanity.unique_non_missing <= 1) | ((sanity.variance > 0) & (sanity.variance <= NEAR_ZERO_VARIANCE_THRESHOLD)) | (sanity.maximum_absolute_value > GROSS_MAGNITUDE_THRESHOLD)]
print("Infinite feature values:", int(infinite_counts.sum()))
print("Constant features:", constant_features)
print("Near-zero variance features:", near_zero_variance_features)
print("Gross-magnitude flags:", gross_magnitude_features)
print("MWL outside nominal 1–10 range:", int((~data.mwl_value.between(1, 10)).sum()))
if len(sanitary_issues):
    print("\nPotential numerical issues:\n", sanitary_issues.to_string(index=False))
if SAVE_TABLES:
    sanity.to_csv(TABLE_DIR / "01_feature_sanity.csv", index=False)
    sanitary_issues.to_csv(TABLE_DIR / "01_feature_sanity_flags.csv", index=False)

## 10. Validation summary

In [ ]:
summary = {
    "rows": len(data),
    "participants": data.participant_id.nunique(),
    "groups": data.group.nunique(),
    "phases": data.phase.nunique(),
    "features": len(feature_columns),
    "duplicate_keys": int(data.duplicated(KEY_COLUMNS).sum()),
    "missing_mwl": int(data.mwl_value.isna().sum()),
    "imputed_mwl": int(data.mwl_source.eq("imputed_previous").sum()),
    "infinite_feature_values": int(infinite_counts.sum()),
    "constant_features": len(constant_features),
    "near_zero_variance_features": len(near_zero_variance_features),
    "blocks_with_partial_modality_coverage": int(modality_trial_coverage.has_partial_modality_coverage.sum()),
}
print("DATASET VALIDATION SUMMARY")
for key, value in summary.items():
    print(f"- {key}: {value}")